[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [18]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.scale = self.d_k ** -0.5


    def forward(self, x, cache=None):
        B, S, d_model = x.size()
        Q = self.W_q(x)  #(B, S, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        Q = Q.view(B, S, self.num_heads, self.d_k).transpose(2, 1)
        K = K.view(B, S, self.num_heads, self.d_k).transpose(2, 1)
        V = V.view(B, S, self.num_heads, self.d_k).transpose(2, 1)

        if cache is not None:
          K_cache, V_cache = cache
          K = torch.concat([K_cache, K], dim=-2)
          V = torch.concat([V_cache, V], dim=-2)
        mask = None
        if S > 1:
          mask = torch.triu(torch.ones(S, S, device=x.device), diagonal=1)
          if cache:
            B, num_heads, total_seq, d_k = K.size()
            mask = torch.concat([torch.zeros(S, total_seq-S, device=x.device), mask], dim=1)
            mask = mask.unsqueeze(0).unsqueeze(0)

        score = Q@K.transpose(-2,-1)*self.scale
        if mask is not None:
          score = score.masked_fill(mask == 1, -float("inf"))
        attn = (score.softmax(dim=-1))@V
        attn = attn.transpose(1, 2).contiguous().view(B, S, d_model)


        attn = self.W_o(attn)

        return attn, (K, V)




        # 1. Project Q, K, V from x
        # 2. Reshape to multi-head: (B, num_heads, S, d_k)
        # 3. If cache exists, concat new K/V with cached K/V
        # 4. Compute attention (causal mask needed during prefill)
        # 5. Return (output, (K_all, V_all))
        pass

In [19]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: True


In [20]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (5.0ms)
  ✅ [2/5] Cache structure (2.3ms)
  ✅ [3/5] Decode step appends to cache (2.1ms)
  ✅ [4/5] Incremental decode matches full forward (2.7ms)
  ✅ [5/5] Gradient flow (2.5ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (14.5ms total)
  Progress saved. Run status() to see your dashboard.

